In [58]:
from FormUtils import pyForm, capture_physics_expr

# 1. FORM: One-Loop Interference and Kinematics

We calculate the interference term between the tree-level (LO) and the one-loop (NLO) diagrams for  
**e⁻ μ⁻ → e⁻ μ⁻**.

## Amplitude Structure
- \( \mathcal{M}_{\text{LO}} \sim e^2 \) (photon exchange)  
- \( \mathcal{M}_{\text{NLO}} \sim e^4 \) (vacuum polarization bubble in the photon propagator)

## Interference
\[
\mathcal{M}_{\text{Int}} = |\mathcal{M}_{\text{LO}} + \mathcal{M}_{\text{NLO}}|^2 
- |\mathcal{M}_{\text{LO}}|^2 
- |\mathcal{M}_{\text{NLO}}|^2 
\approx 2 \, \mathrm{Re}(\mathcal{M}_{\text{LO}}^\dagger \mathcal{M}_{\text{NLO}})
\]

## Massless Approximation
We work in the limit:
\[
M_{\mu}, M_{e} \to 0
\]
for the kinematics, while retaining \( M_e \) inside the loop propators.

## Kinematic Mapping
- \( k_{f1} = x \cdot q \)  
- \( k_{f2} = (x - 1)\cdot q \)  
- \( t = q^2 \)

---

# 2. Feynman Parametrization and the Master Integral

\[
\frac{1}{AB} = \int_0^1 \frac{dx}{\left[xA + (1-x)B\right]^2}
\]

## Master Integral

\[
I(t, m^2) = \frac{1}{16\pi^2} \int_0^1 dx \, \ln\left(\frac{m^2 - x(1-x)t}{\Lambda^2}\right)
\]

---

# 3. Python: 1D Integration and Numerical Extraction

## Kernel

\[
\text{Kernel} = \frac{\mathcal{M}_{\text{Int}}(x)}{\mathcal{M}_{\text{LO}}}
\]

## Integration

\[
\Delta = \int_0^1 \text{Kernel}(x)\,\text{MasterProp}(x, t, \Lambda, M_e)\, dx
\]

---

# 4. The Relative Correction

\[
\Delta \approx \frac{e^2}{6\pi^2} \ln\left(\frac{\Lambda}{M_e}\right) + \text{const}
\]

In [59]:
%%pyForm Running

* Process: Running

#-
* Above suppresses extra output
Off Statistics;
Off FinalStats;
#include SquareAmplitude.h

Symbols e, Mmuon, Melec;
Vectors k, kf1, kf2;
Symbols MasterProp, fx;


Local MLO = (e^2) * (UB(i1, p3, Melec ) * g(i1, i2, mu1) * U(i2, p1, Melec))
            * phprop(mu1, mu2, q)
            * (UB(i3, p4, Mmuon) * g(i3, i4, mu2) * U(i4, p2, Mmuon));

Local MNLO = -1* (e^4) 
             * (UB(i1, p3, Melec) * g(i1, i2, mu1) * U(i2, p1, Melec)) 
             * phprop(mu1, mu3, q) 
             * g(i11, i12, mu3) * fprop(i12, i13, kf1, Melec) 
             * g(i13, i14, mu4) * fprop(i14, i11, kf2, Melec)
             * phprop(mu4, mu2, q)
             * (UB(i7, p4, Mmuon) * g(i7, i8, mu2) * U(i8, p2, Mmuon));

Local MTotal = MLO+MNLO;
#call squareamplitude(MLO, MsqLO)
#call squareamplitude(MNLO, MsqNLO)
#call squareamplitude(MTotal, MsqTotal)
.sort
Multiply 1/4;
.sort
Drop drop MsqNLO, MsqTotal, MLO, MNLO, MTotal ;
Local MInt = MsqTotal - MsqLO - MsqNLO;
.sort

* --- KINEMATIC DEFINITIONS: VACUUM POLARIZATION ---
* q  = p1 - p3           : Momentum transfer between electron and muon lines
* t  = q.q               : Mandelstam variable t (photon momentum squared)
* k  = loop momentum     : Integration variable for the fermion loop
* kf1 = k                : Momentum of the first fermion in the bubble
* kf2 = k - q            : Momentum of the second fermion in the bubble
* Replace the propagator function
id prop(q.q) = 1/t;
id prop(-Melec^2 + kf1.kf1) * prop(-Melec^2 + kf2.kf2) = MasterProp;
.sort
* kf1 effectively becomes fx * q
id kf1.p1 = fx * (t/2);
id kf1.p2 = fx * (-t/2);
id kf1.p3 = fx * (-t/2);
id kf1.p4 = fx * (t/2);
* kf2 effectively becomes (fx - 1) * q
id kf2.p1 = (fx - 1) * (t/2);
id kf2.p2 = (fx - 1) * (-t/2);
id kf2.p3 = (fx - 1) * (-t/2);
id kf2.p4 = (fx - 1) * (t/2);
* Internal loop propagator dot product
id kf1.kf2 = fx * (fx - 1) * t;
.sort
id Melec = 0;
id Mmuon = 0;
* --- MASSLESS APPROXIMATION ---
#call Mandelstam2To2(p1,p2,p3,p4,0,0,0,0)
.sort
id u = -s -t;
.sort
 
* Print 
Format;
Print+s MInt;
Print+s MsqLO;
* Save to file 
Format C;
#write <RunningLO.txt> "%e;", MsqLO;
#write <Running.txt> "%e;", MInt;
.sort

.end

FORM 5.0.0 (Jan 27 2026, v5.0.0)                 Run: Fri Apr 17 19:58:37 2026
    
    * Process: Running
    
    #-

   MsqLO =
       + 2*pow(e,4)
       + 4*s*pow(t,-1)*pow(e,4)
       + 4*pow(s,2)*pow(t,-2)*pow(e,4)
      ;

   MInt =
       + 16*pow(e,6)*MasterProp*fx
       - 16*pow(e,6)*MasterProp*pow(fx,2)
       + 32*s*pow(t,-1)*pow(e,6)*MasterProp*fx
       - 32*s*pow(t,-1)*pow(e,6)*MasterProp*pow(fx,2)
       + 32*pow(s,2)*pow(t,-2)*pow(e,6)*MasterProp*fx
       - 32*pow(s,2)*pow(t,-2)*pow(e,6)*MasterProp*pow(fx,2)
      ;




In [60]:
import numpy as np
import sympy as sp

form_expr_LO = capture_physics_expr("scripts/RunningLO.txt")
form_expr_Int = capture_physics_expr("scripts/Running.txt")

t, s, u, e = sp.symbols("t s u e", real=True)
fx = sp.Symbol("fx", real=True)
MasterProp = sp.Symbol("MasterProp")

mapping = {
    "fx": fx,
    "s": s,
    "t": t,
    "u": u,
    "e": e,
    "MasterProp": MasterProp,
}

MLO = sp.parse_expr(form_expr_LO, local_dict=mapping)
MInt = sp.parse_expr(form_expr_Int, local_dict=mapping)
print("MLO :")
sp.pprint(MLO)
print("\n")
print("MInt")
sp.pprint(MInt)
print("\n")


MInt_num = MInt.subs({MasterProp: 1})
print("MInt_num")
sp.pprint(MInt_num)
print("\n")
MInt_clean = sp.cancel(MInt_num)
poly_fx = sp.Poly(MInt_clean, fx)
a1 = poly_fx.nth(1)
a2 = poly_fx.nth(2)
print("Coefficient a1 (Kinematics):")
sp.pprint(sp.simplify(a1))
print("\nCoefficient a2 (Kinematics):")
sp.pprint(sp.simplify(a2))

# Extract the Relative Correction Kernel (delta_kernel)
ratio_kernel = sp.simplify((a1 * fx + a2 * fx**2) / MLO)
print("Ratio Kernel:")
sp.pprint(ratio_kernel)

Melec = sp.symbols("Melec", real=True, positive=True)
Lambda = sp.symbols("Lambda", real=True, positive=True)
MasterProp_Integrated = (1 / (16 * sp.pi**2)) * sp.log(
    Lambda**2 / (Melec**2 - fx * (1 - fx) * t)
)

integral1D = sp.integrate(
    ratio_kernel * MasterProp_Integrated, (fx, 0, 1)
)  # Result: 4/3 * e^2
print("integral1D :")
sp.pprint(integral1D)
res_expanded = sp.series(integral1D, Melec, 0, 1).removeO()
print("Result:")
sp.pprint(res_expanded)
coeff_log_lambda = res_expanded.coeff(sp.log(Lambda))

MLO :
   4  2      4         
4⋅e ⋅s    4⋅e ⋅s      4
─────── + ────── + 2⋅e 
   2        t          
  t                    


MInt
                 6   2  2                  6   2                               ↪
  32⋅MasterProp⋅e ⋅fx ⋅s    32⋅MasterProp⋅e ⋅fx ⋅s                  6   2   32 ↪
- ─────────────────────── - ────────────────────── - 16⋅MasterProp⋅e ⋅fx  + ── ↪
             2                        t                                        ↪
            t                                                                  ↪

↪              6     2                  6                           
↪ ⋅MasterProp⋅e ⋅fx⋅s    32⋅MasterProp⋅e ⋅fx⋅s                  6   
↪ ──────────────────── + ───────────────────── + 16⋅MasterProp⋅e ⋅fx
↪          2                       t                                
↪         t                                                         


MInt_num
      6   2  2       6   2                     6     2       6                
  32⋅e ⋅fx ⋅s    32⋅e ⋅fx 